# MemBukkit Quickstart

Ingest conversations, ask questions with dated answers, inspect *why* each answer was produced, and persist everything to a local store.

You need an `OPENAI_API_KEY` in your environment (or use `llm="ollama:llama3.1"` for a zero-cost local setup). Model weights download automatically on first use.

In [ ]:
# Install (run once)
# !pip install "membukkit[all]"

## 1. Basic Usage

Create a MemorySystem, ingest some conversations, and ask questions.

In [ ]:
from membukkit import MemorySystem, RetrievalConfig

# Weights auto-download from the HF Hub on first use (set MEMBUKKIT_MODEL_DIR to override).
# For tiny corpora like this notebook, scan everything — the scan budget is a
# large-corpus efficiency lever (see notebook 02).
mem = MemorySystem.from_pretrained(
    retrieval=RetrievalConfig(scan_budget=1.0, scan_budget_reason=1.0),
    llm="openai:gpt-4o-mini",
)

In [ ]:
# Ingest conversations
sessions = [
    [
        {"role": "user", "content": "I just adopted a dog named Luna from the shelter!"},
        {"role": "assistant", "content": "Congratulations! What breed is Luna?"},
        {"role": "user", "content": "She's a golden retriever mix, about 2 years old."},
    ],
    [
        {"role": "user", "content": "Luna is settling in well. We've been going to Riverside Park every morning."},
        {"role": "assistant", "content": "That sounds lovely! Does she enjoy it?"},
        {"role": "user", "content": "She loves it. We also signed up for training classes at PetSmart."},
    ],
]

n_new = mem.ingest(sessions=sessions, dates=["2024-01-15", "2024-02-01"])
print(f"Ingested {n_new} facts ({mem.backend.count_kind('verbatim')} verbatim + {mem.backend.count_kind('atomic')} atomic)")

In [ ]:
# Ask a question
result = mem.answer("When did I adopt my dog?", question_date="2024/06/01")
print(f"Answer: {result.answer}")
print(f"Scan fraction: {result.trace.scan_fraction:.1%}")
print(f"Reader type: {result.trace.reader_type}")
print(f"\nRetrieved facts:")
for f in result.facts:
    print(f"  {f}")

## 2. Inspect Buckets

The partition is inspectable — you can see which facts ended up in which bucket.

In [ ]:
partition = mem.partition()
print(f"Number of buckets: {partition.get('k_eff', 0)}")
print(f"Facts per bucket: {', '.join(f'{k}:{len(v)}' for k, v in partition.get('by_bucket', {}).items())}")

In [ ]:
# Auto-label buckets
labels = mem.label_buckets()
for b, label in labels.items():
    print(f"Bucket {b}: {label}")

## 3. Persist to a local store

Save the memory bank to `~/.membukkit/stores/<name>` and reload it any time — the CLI (`membukkit ask --store pets`) and the GUI read the same stores.

In [ ]:
from membukkit.storage import LocalStore

store = LocalStore("pets")
store.save_backend(mem.backend)
print(f"saved {store.meta()['n_facts']} facts to {store.dir}")

# ...and reload into a fresh system
mem2 = MemorySystem.from_pretrained(llm="openai:gpt-4o-mini",
                                    retrieval=RetrievalConfig(scan_budget=1.0))
store.load_backend(mem2.backend)
print(mem2.answer("Where do we walk Luna?", question_date="2024-06-01").answer)

## 4. Configuration

All knobs are exposed through dataclass configs.

In [ ]:
from dataclasses import asdict
from membukkit.config import RetrievalConfig

rc = RetrievalConfig(
    num_buckets=12,        # fewer buckets for smaller banks
    scan_budget=0.5,       # scan more for higher recall
    select="cosine",       # simpler selection mode
    top_k=20,              # more facts to reader
)
print(asdict(rc))